In [ ]:
# 0) Environment + reusable runner (run first)
from pathlib import Path
import os, sys, json, subprocess, shutil
from datetime import datetime

_cwd = Path.cwd().resolve()
_candidates = [
    _cwd,
    _cwd.parent,
    _cwd / 'backend',
    _cwd.parent / 'backend',
    Path('/content/CausalX-Project/backend'),
    Path('/content/drive/MyDrive/CausalX-Project/backend'),
]
PROJECT_ROOT = next((p for p in _candidates if (p / 'src').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate backend root with src/ directory.')

def _load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = os.path.expandvars(os.path.expanduser(v))

_load_env_file(PROJECT_ROOT / 'configs' / 'dataset_paths.env')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VENV_PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
PY_BIN = str(VENV_PY if VENV_PY.exists() else Path(sys.executable))

os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
os.environ.setdefault('CFN_USE_EMBEDDINGS', 'true')
os.environ.setdefault('CFN_W2V2_MODEL', 'WAV2VEC2_BASE')
os.environ.setdefault('CFN_EMB_MODEL_PATH', str(PROJECT_ROOT / 'models' / 'cfn_emb.pth'))
os.environ.setdefault('CFN_VISUAL_TCN_PATH', str(PROJECT_ROOT / 'models' / 'visual_tcn.pth'))
os.environ.setdefault('MEDIAPIPE_DISABLE_GPU', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = PROJECT_ROOT / 'models' / 'experiment_logs' / f'mixed_eval_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)


def run_cmd(cmd, name, extra_env=None, check=True):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    print("
$ " + ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, text=True, capture_output=True)

    (RUN_DIR / f'{name}.stdout.log').write_text(res.stdout or '')
    (RUN_DIR / f'{name}.stderr.log').write_text(res.stderr or '')
    (RUN_DIR / f'{name}.meta.json').write_text(json.dumps({'cmd': cmd, 'returncode': res.returncode}, indent=2))

    if res.stdout:
        print(res.stdout[-4000:])
    if res.stderr:
        print('--- stderr ---')
        print(res.stderr[-4000:])

    if check and res.returncode != 0:
        raise RuntimeError(f'{name} failed with code {res.returncode}')
    return res


def snapshot_model(tag):
    src_model = PROJECT_ROOT / 'models' / 'cfn_emb.pth'
    src_scaler = PROJECT_ROOT / 'models' / 'cfn_scaler.pkl'
    if src_model.exists():
        shutil.copy2(src_model, RUN_DIR / f'{tag}.cfn_emb.pth')
    if src_scaler.exists():
        shutil.copy2(src_scaler, RUN_DIR / f'{tag}.cfn_scaler.pkl')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PY_BIN:', PY_BIN)
print('RUN_DIR:', RUN_DIR)


## Stage A — Baseline mixed-domain retraining


In [ ]:
# 1) Retrain A
run_cmd([
    PY_BIN, '-m', 'src.training.train_cfn',
    '--data', 'data/processed/causal_multimodal_dataset.csv',
    '--train-source', 'all',
    '--use-embeddings', '--use-scaler',
    '--group-balance', '--use-weighted-sampler',
    '--loss', 'focal', '--focal-alpha', '0.75', '--focal-gamma', '2.0',
    '--causal-weight', '0.15',
    '--scheduler', 'cosine',
    '--epochs', '35', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4',
    '--selection-metric', 'hybrid_robust',
    '--selection-threshold', '0.5',
    '--min-domain-spec', '0.30',
    '--min-domain-rec', '0.50',
], '01_train_A')
snapshot_model('01_train_A')


## Stage B — Hard-negative retraining


In [ ]:
# 3) Mine hard negatives from result+cache
import numpy as np

def mine_hard_negatives(result_json, cache_json, out_tsv):
    res = json.loads(Path(result_json).read_text())
    cache = json.loads(Path(cache_json).read_text())

    p = float(res['best']['prob'])
    r = float(res['best']['ratio'])
    c = float(res['best']['causal'])
    rf = bool(res['best']['require_flag'])

    out_tsv = Path(out_tsv)
    out_tsv.parent.mkdir(parents=True, exist_ok=True)

    n = 0
    with out_tsv.open('w') as f:
        for row in cache:
            y = int(row['label'])
            probs = np.array(row.get('probs', []), dtype=float)
            mism = np.array(row.get('mism', []), dtype=float)
            if probs.size == 0:
                pred = 0
            else:
                suspicious = ((probs >= p) & (mism >= c)) if rf else ((probs >= p) | (mism >= c))
                pred = int(float(np.mean(suspicious)) >= r)
            if y == 0 and pred == 1:
                f.write(f"{row['path']}\t0\t1\n")
                n += 1
    return n

hn_b = PROJECT_ROOT / 'data' / 'processed' / 'hard_negatives_mixed.tsv'
n_b = mine_hard_negatives(
    PROJECT_ROOT / 'models' / 'mixed_sweep_results_v2_constrained.json',
    PROJECT_ROOT / 'eval_cache_mixed_v2.json',
    hn_b,
)
print('Hard negatives B:', n_b, '->', hn_b)


In [ ]:
# 4) Retrain B (hard negatives)
run_cmd([
    PY_BIN, '-m', 'src.training.train_cfn',
    '--data', 'data/processed/causal_multimodal_dataset.csv',
    '--train-source', 'all',
    '--use-embeddings', '--use-scaler',
    '--group-balance', '--use-weighted-sampler',
    '--hard-negative-file', 'data/processed/hard_negatives_mixed.tsv',
    '--hard-negative-weight', '4.0',
    '--loss', 'focal', '--focal-alpha', '0.75', '--focal-gamma', '2.0',
    '--causal-weight', '0.15',
    '--scheduler', 'cosine',
    '--epochs', '40', '--patience', '10', '--batch-size', '128',
    '--lr', '2e-4', '--weight-decay', '1e-4',
    '--selection-metric', 'hybrid_robust',
    '--selection-threshold', '0.5',
    '--min-domain-spec', '0.30',
    '--min-domain-rec', '0.50',
], '03_train_B')
snapshot_model('03_train_B')


## Stage D — Re-mine hard negatives from C and retrain once more


In [ ]:
# 13) Final summary + env apply + evidence report
best = SELECTED_PAYLOAD['best']
overall = best['metrics']['overall']
per_ds = best['metrics']['per_dataset']

print('==== FINAL MIXED RESULT ====')
print('Best config:')
print('  PROB=', best['prob'], 'RATIO=', best['ratio'], 'CAUSAL=', best['causal'], 'REQUIRE_FLAG=', best['require_flag'])
print('Overall:')
print('  Acc={acc:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} Rec={rec:.3f} Spec={spec:.3f}'.format(**overall))
print('Per dataset:')
for ds, m in per_ds.items():
    print(f"  [{ds}] Acc={m['acc']:.3f} BalAcc={m['bal_acc']:.3f} F1={m['f1']:.3f} Rec={m['rec']:.3f} Spec={m['spec']:.3f}")

print("\nRecommended env vars:")
for k, v in SELECTED_PAYLOAD['recommend_env'].items():
    print(f'{k}={v}')
    os.environ[k] = v

report = {
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'selected_result_file': selected['path'],
    'recommend_env': SELECTED_PAYLOAD['recommend_env'],
    'overall': overall,
    'per_dataset': per_ds,
    'macro_bal_acc': best['metrics']['macro_bal_acc'],
    'macro_f1': best['metrics']['macro_f1'],
}
report_path = RUN_DIR / 'final_report.json'
report_path.write_text(json.dumps(report, indent=2))
print("\nSaved evidence report:", report_path)


In [ ]:
# 14) Train video-level decision calibrator from selected sweep cache
selected_cache = SELECTED_PAYLOAD.get('cache')
if not selected_cache:
    raise RuntimeError('Selected payload does not include cache path.')

run_cmd([
    PY_BIN, 'scripts/train_video_calibrator.py',
    '--cache', selected_cache,
    '--out', 'models/video_calibrator.pkl',
    '--min-rec', '0.70',
    '--min-spec', '0.45',
], '10_train_video_calibrator')


In [ ]:
# 15) Apply calibrator in runtime + persist env recommendation
import joblib

cal_payload = joblib.load(PROJECT_ROOT / 'models' / 'video_calibrator.pkl')
cal_thr = float(cal_payload.get('threshold', 0.5))
os.environ['CFN_VIDEO_CALIBRATOR_PATH'] = str(PROJECT_ROOT / 'models' / 'video_calibrator.pkl')
os.environ['CFN_CALIBRATOR_THRESH'] = f'{cal_thr:.4f}'

print('Applied calibrator env:')
print('CFN_VIDEO_CALIBRATOR_PATH=', os.environ['CFN_VIDEO_CALIBRATOR_PATH'])
print('CFN_CALIBRATOR_THRESH=', os.environ['CFN_CALIBRATOR_THRESH'])


## Stage E — Clean mixed train/val/test split (no threshold leakage)

This stage creates split-specific CSVs/manifests, tunes thresholds on **val**, then reports final metrics on **test** using fixed val-selected thresholds.


In [ ]:
# 20) Print final clean test metrics + env recommendation
test_payload = json.loads((PROJECT_ROOT / 'models' / 'mixed_test_eval_fixed_from_val.json').read_text())
b = test_payload['best']
overall = b['metrics']['overall']
per_ds = b['metrics']['per_dataset']

print('==== CLEAN TEST RESULT (thresholds selected on VAL) ====')
print('Best config (from VAL):')
print('  PROB=', b['prob'], 'RATIO=', b['ratio'], 'CAUSAL=', b['causal'], 'REQUIRE_FLAG=', b['require_flag'])
print('Overall:')
print('  Acc={acc:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} Rec={rec:.3f} Spec={spec:.3f}'.format(**overall))
print('Per dataset:')
for ds, m in per_ds.items():
    print(f"  [{ds}] Acc={m['acc']:.3f} BalAcc={m['bal_acc']:.3f} F1={m['f1']:.3f} Rec={m['rec']:.3f} Spec={m['spec']:.3f}")

print('Recommended env vars:')
for k, v in test_payload['recommend_env'].items():
    print(f'{k}={v}')


## Stage F — Dual error mining + 3-seed ensemble

Adds two robustness methods:
- Dual error mining (FP + FN) and upweight retraining
- 3-seed ensemble evaluation on clean test split


In [ ]:
# 21) Mine dual error paths (FP + FN) from VAL-selected thresholds
val_payload = json.loads((PROJECT_ROOT / 'models' / 'mixed_sweep_val_split.json').read_text())
v = val_payload['best']

run_cmd([
    PY_BIN, 'scripts/mine_error_paths.py',
    '--cache', 'eval_cache_mixed_val.json',
    '--prob', str(v['prob']),
    '--ratio', str(v['ratio']),
    '--causal', str(v['causal']),
    '--require-flag', str(v['require_flag']).lower(),
    '--out-fp', 'data/processed/error_fp_val.tsv',
    '--out-fn', 'data/processed/error_fn_val.tsv',
    '--out-combined', 'data/processed/error_combined_val.tsv',
], '15_mine_dual_errors')


In [ ]:
# Apply selected VAL thresholds and evaluate on CLEAN TEST cache
import json
import numpy as np
from pathlib import Path

SEL = json.loads((PROJECT_ROOT / "models" / "val_sweep_real_guard.json").read_text())
env = SEL["recommend_env"]
print("Using:", env)

TEST_CACHE = PROJECT_ROOT / "eval_cache_mixed_test.json"
test_cache = json.loads(TEST_CACHE.read_text())
print("TEST rows:", len(test_cache))

p = float(env["CFN_PROB_THRESH"])
r = float(env["CFN_RATIO_THRESH"])
c = float(env["CFN_CAUSAL_THRESH"])
rf = env["CFN_REQUIRE_FLAG"].lower() == "true"

def conf(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    acc  = (tp + tn) / total if total else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    bal  = 0.5*(rec + spec)
    f1   = 2*prec*rec/(prec + rec) if (prec + rec) else 0.0
    return {"acc":acc, "prec":prec, "rec":rec, "spec":spec, "bal_acc":bal, "f1":f1}

by_ds = {}
T = {"tp":0, "tn":0, "fp":0, "fn":0}
for row in test_cache:
    ds = row.get("dataset", "Unknown")
    if ds not in by_ds:
        by_ds[ds] = {"tp":0, "tn":0, "fp":0, "fn":0}
    y = int(row["label"])
    probs = np.array(row["probs"], dtype=float)
    mism  = np.array(row["mism"], dtype=float)

    if probs.size == 0:
        pred = 0
    else:
        suspicious = ((probs >= p) & (mism >= c)) if rf else ((probs >= p) | (mism >= c))
        pred = int(float(np.mean(suspicious)) >= r)

    if pred == 1 and y == 1:
        by_ds[ds]["tp"] += 1; T["tp"] += 1
    elif pred == 0 and y == 0:
        by_ds[ds]["tn"] += 1; T["tn"] += 1
    elif pred == 1 and y == 0:
        by_ds[ds]["fp"] += 1; T["fp"] += 1
    else:
        by_ds[ds]["fn"] += 1; T["fn"] += 1

overall = conf(T["tp"], T["tn"], T["fp"], T["fn"])
print("==== CLEAN TEST RESULT (VAL-CONSTRAINED) ====")
print("Overall:", overall)
for ds, cm in by_ds.items():
    print(ds, conf(cm["tp"], cm["tn"], cm["fp"], cm["fn"]))


In [ ]:
!.venv/bin/python -m src.training.train_cfn \
  --data data/processed/causal_multimodal_dataset_mixed_train_balanced.csv \
  --train-source all \
  --use-embeddings --use-scaler \
  --group-balance --use-weighted-sampler \
  --hard-negative-file data/processed/error_combined_val.tsv \
  --hard-negative-weight 6.0 \
  --loss focal --focal-alpha 0.75 --focal-gamma 2.0 \
  --causal-weight 0.15 \
  --scheduler cosine \
  --epochs 50 --patience 12 --batch-size 128 \
  --lr 2e-4 --weight-decay 1e-4 \
  --seed 42 \
  --selection-metric hybrid_robust \
  --selection-threshold 0.5 \
  --min-domain-spec 0.25 \
  --min-domain-rec 0.45
